# Vesuvius Challenge 2025 - Topology-Aware 3D Segmentation V2

## Target Score: 0.60-0.75+

### Key Improvements over Baseline:
| Component | Baseline (0.443) | This Version |
|-----------|------------------|---------------|
| Patch Size | 64×128×128 | **192×192×192** |
| Model | 5-stage UNet | **6-stage ResEncUNet + scSE** |
| Epochs | 82 | **1200** |
| Loss | Dice only | **Dice + BCE + SkeletonRecall + soft-clDice** |
| Post-proc | threshold 0.09 | **Skeleton-guided** |

### References:
- [Skeleton Recall Loss (ECCV 2024)](https://github.com/MIC-DKFZ/Skeleton-Recall)
- [soft-clDice (CVPR 2021)](https://github.com/jocpae/clDice)
- [Betti Matching 3D](https://github.com/nstucki/Betti-Matching-3D)

In [1]:
!pip install imagecodecs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 94.9 MB/s eta 0:00:00:00:0100:01


In [ ]:
# =============================================================================
# CELL 1: IMPORTS & CONFIGURATION
# =============================================================================

import os
import gc
import json
import random
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass

import numpy as np
import pandas as pd
from PIL import Image, ImageSequence
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from scipy import ndimage
from scipy.ndimage import distance_transform_edt
from skimage.morphology import skeletonize

try:
    import cc3d
    USE_CC3D = True
except ImportError:
    USE_CC3D = False
    print("cc3d not found, using scipy for connected components")

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION FOR H100 (80GB GPU + 220GB RAM)
# =============================================================================

@dataclass
class Config:
    # Data paths
    DATA_ROOT: Path = Path("/kaggle/input/vesuvius-challenge-surface-detection")
    CHECKPOINT_DIR: Path = Path("/kaggle/working/checkpoints")

    # Keep many periodic checkpoints for SWA / averaging (set lower if disk is tight)
    MAX_KEEP_CHECKPOINTS: int = 9999
    
    # Fine-tune focus (single scroll to match test)
    TARGET_SCROLL_ID: int = 26002
    TEST_ID: str = "1407735"
    VAL_N: int = 10

    # Fine-tune data selection
    # 'scroll': only volumes from TARGET_SCROLL_ID (fastest adaptation, may overfit if private differs)
    # 'all':    all training volumes except TEST_ID (safest generalization)
    FINETUNE_DATA_MODE: str = "scroll"  # 'scroll' or 'all'
    GLOBAL_VAL_FRACTION: float = 0.2     # used when FINETUNE_DATA_MODE='all'
    
    # Fine-tune from checkpoint weights only (fresh optimizer/scheduler)
    FINETUNE_WEIGHTS_ONLY: bool = True
    INIT_CHECKPOINT_PATH: Optional[Path] = None  # set this to a .pth checkpoint path
    
    # Validation mode
    USE_FULL_VOLUME_VAL: bool = False
    VAL_THRESHOLD: float = 0.5
    VAL_OVERLAP: float = 0.7
    
    # Model architecture
    PATCH_SIZE: Tuple[int, int, int] = (192, 192, 192)
    FEATURES: List[int] = None
    N_BLOCKS: List[int] = None
    USE_SCSE: bool = True
    USE_DEEP_SUPERVISION: bool = True
    
    # Training settings
    EPOCHS: int = 60
    BATCH_SIZE: int = 4
    NUM_WORKERS: int = 8
    LEARNING_RATE: float = 0.01
    MOMENTUM: float = 0.99
    WEIGHT_DECAY: float = 3e-5
    GRADIENT_CLIP: float = 12.0
    
    # Loss weights (used when FINETUNE_MODE=False)
    DICE_WEIGHT: float = 0.3
    BCE_WEIGHT: float = 0.2
    SKELETON_WEIGHT: float = 0.25
    CLDICE_WEIGHT: float = 0.25
    
    # Loss scheduling
    SKELETON_START_EPOCH: int = 300
    SKELETON_WARMUP_EPOCHS: int = 300
    CLDICE_START_EPOCH: int = 600
    CLDICE_WARMUP_EPOCHS: int = 300
    
    # Training control
    RESUME_TRAINING: bool = False
    VALIDATE_EVERY: int = 10
    SAVE_EVERY: int = 5
    
    # ==========================================================================
    # LR FIX: Override LR from checkpoint with a specific value
    # ==========================================================================
    OVERRIDE_LR: float = 0.0005  # Fine-tuning LR (lower than before for stability)
    # ==========================================================================
    
    # Legacy (keep False)
    REDUCE_LR_NOW: bool = False
    LR_REDUCTION_FACTOR: float = 0.5
    AUTO_LR_REDUCTION_AT_TOPOLOGY: bool = False
    
    # ==========================================================================
    # FINE-TUNING MODE: Adds Boundary Loss + Hausdorff DT Loss
    # ==========================================================================
    # Enable this to add boundary-aware losses on top of existing losses.
    # These sharpen boundaries and improve topology without changing architecture.
    # The dataset will also precompute distance transforms for boundary loss.
    FINETUNE_MODE: bool = True
    BOUNDARY_WEIGHT: float = 0.15       # Weight for boundary loss
    HAUSDORFF_WEIGHT: float = 0.10      # Weight for Hausdorff DT loss
    BOUNDARY_WARMUP_EPOCHS: int = 20    # Epochs to linearly ramp boundary losses
    # When FINETUNE_MODE=True, base weights are rebalanced:
    FT_DICE_WEIGHT: float = 0.20
    FT_BCE_WEIGHT: float = 0.15
    FT_SKELETON_WEIGHT: float = 0.20
    FT_CLDICE_WEIGHT: float = 0.20
    # Total: 0.20 + 0.15 + 0.20 + 0.20 + 0.15 + 0.10 = 1.0
    # ==========================================================================
    
    # Device
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP: bool = True
    
    # Data loading
    DATA_FRACTION: float = 1.0
    PATCHES_PER_VOLUME: int = 8
    PRELOAD_ALL: bool = True
    
    # Random seed
    SEED: int = 42
    
    def __post_init__(self):
        if self.FEATURES is None:
            self.FEATURES = [32, 64, 128, 256, 320, 320]
        if self.N_BLOCKS is None:
            self.N_BLOCKS = [1, 3, 4, 6, 6, 6]
        self.CHECKPOINT_DIR = Path(self.CHECKPOINT_DIR)
        self.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        if self.INIT_CHECKPOINT_PATH is not None:
            self.INIT_CHECKPOINT_PATH = Path(self.INIT_CHECKPOINT_PATH)

cfg = Config()
cfg.__post_init__()

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True

set_seed(cfg.SEED)

print("="*60)
print("H100 CONFIGURATION")
print("="*60)
print(f"Batch size: {cfg.BATCH_SIZE}")
print(f"Workers: {cfg.NUM_WORKERS}")
print(f"Patches/volume: {cfg.PATCHES_PER_VOLUME}")
if cfg.OVERRIDE_LR > 0:
    print(f"OVERRIDE_LR: {cfg.OVERRIDE_LR} (will force this LR on resume)")
if cfg.FINETUNE_MODE:
    print(f"FINETUNE_MODE: ON")
    print(f"  Boundary weight: {cfg.BOUNDARY_WEIGHT}")
    print(f"  Hausdorff weight: {cfg.HAUSDORFF_WEIGHT}")
    print(f"  Warmup: {cfg.BOUNDARY_WARMUP_EPOCHS} epochs")
print("="*60)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")


In [ ]:
# =============================================================================
# CELL 1B: TRAIN/VAL SPLIT (NO LEAKAGE)
# =============================================================================
# Kaggle uses only the IDs in test.csv for evaluation. If you still want to be
# extra-safe for private split/generalization, you can fine-tune on all volumes.


def make_scroll_split(
    csv_path: Path,
    target_scroll_id: int,
    test_id: str,
    seed: int = 42,
    val_n: int = 10,
):
    df = pd.read_csv(csv_path)
    ids = df.loc[
        (df['scroll_id'] == target_scroll_id) & (df['id'].astype(str) != str(test_id)),
        'id',
    ].astype(str).tolist()
    ids = sorted(ids)
    rng = np.random.RandomState(seed)
    rng.shuffle(ids)

    if len(ids) == 0:
        raise ValueError(f"No training ids found for scroll_id={target_scroll_id}. Check cfg.DATA_ROOT.")

    val_n = min(int(val_n), len(ids) - 1) if len(ids) > 1 else 1
    val_ids = ids[:val_n]
    train_ids = ids[val_n:]
    return train_ids, val_ids


def make_all_split(
    csv_path: Path,
    test_id: str,
    seed: int = 42,
    val_fraction: float = 0.2,
):
    df = pd.read_csv(csv_path)
    ids = df.loc[df['id'].astype(str) != str(test_id), 'id'].astype(str).tolist()
    ids = sorted(ids)
    rng = np.random.RandomState(seed)
    rng.shuffle(ids)

    if len(ids) == 0:
        raise ValueError("No training ids found. Check cfg.DATA_ROOT.")

    val_n = int(round(len(ids) * float(val_fraction)))
    val_n = max(1, val_n)
    val_n = min(val_n, len(ids) - 1) if len(ids) > 1 else 1

    val_ids = ids[:val_n]
    train_ids = ids[val_n:]
    return train_ids, val_ids


train_csv_path = cfg.DATA_ROOT / "train.csv"

if cfg.FINETUNE_DATA_MODE == "scroll":
    cfg.TRAIN_VOLUME_IDS, cfg.VAL_VOLUME_IDS = make_scroll_split(
        train_csv_path,
        target_scroll_id=cfg.TARGET_SCROLL_ID,
        test_id=cfg.TEST_ID,
        seed=cfg.SEED,
        val_n=cfg.VAL_N,
    )
    print(
        f"Mode=scroll({cfg.TARGET_SCROLL_ID}): train={len(cfg.TRAIN_VOLUME_IDS)} val={len(cfg.VAL_VOLUME_IDS)} "
        f"(excluded test_id={cfg.TEST_ID})"
    )

elif cfg.FINETUNE_DATA_MODE == "all":
    cfg.TRAIN_VOLUME_IDS, cfg.VAL_VOLUME_IDS = make_all_split(
        train_csv_path,
        test_id=cfg.TEST_ID,
        seed=cfg.SEED,
        val_fraction=cfg.GLOBAL_VAL_FRACTION,
    )
    print(
        f"Mode=all: train={len(cfg.TRAIN_VOLUME_IDS)} val={len(cfg.VAL_VOLUME_IDS)} "
        f"(excluded test_id={cfg.TEST_ID}) val_fraction={cfg.GLOBAL_VAL_FRACTION}"
    )

else:
    raise ValueError(f"Unknown cfg.FINETUNE_DATA_MODE={cfg.FINETUNE_DATA_MODE!r} (use 'scroll' or 'all')")

print("Val IDs:", cfg.VAL_VOLUME_IDS)
print(
    f"Fine-tune epochs: {cfg.EPOCHS} | LR: {cfg.OVERRIDE_LR} | SAVE_EVERY: {cfg.SAVE_EVERY} | VALIDATE_EVERY: {cfg.VALIDATE_EVERY}"
)


In [ ]:
# =============================================================================
# CELL 2: CHECKPOINT MANAGER
# =============================================================================

class CheckpointManager:
    """Manages saving and loading of training checkpoints."""
    
    def __init__(self, save_dir: Path, load_dir: Path = None, max_keep: int = 5):
        self.save_dir = Path(save_dir)
        self.load_dir = Path(load_dir) if load_dir else self.save_dir
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.max_keep = max_keep
        self.best_score = -1
        self.best_epoch = -1
        self.history = []
    
    def save(self, model, optimizer, scheduler, scaler, epoch: int, metrics: dict):
        """Save checkpoint with all training state."""
        ckpt = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
            'scaler_state_dict': scaler.state_dict() if scaler else None,
            'metrics': metrics,
            'best_score': self.best_score,
            'best_epoch': self.best_epoch,
            'config': {
                'features': cfg.FEATURES,
                'n_blocks': cfg.N_BLOCKS,
                'patch_size': cfg.PATCH_SIZE,
            }
        }
        
        # Save last checkpoint (for resuming)
        torch.save(ckpt, self.save_dir / 'last_checkpoint.pth')
        
        # Check for best model - ONLY when validation actually ran (val_dice > 0)
        val_dice = metrics.get('val_dice', 0)
        if val_dice > 0 and val_dice > self.best_score:
            self.best_score = val_dice
            self.best_epoch = epoch
            ckpt['best_score'] = self.best_score
            ckpt['best_epoch'] = self.best_epoch
            torch.save(ckpt, self.save_dir / 'best_model.pth')
            print(f"  >>> New best model! Val Dice: {val_dice:.4f}")
        
        # Save periodic checkpoint
        if (epoch + 1) % cfg.SAVE_EVERY == 0:
            torch.save(ckpt, self.save_dir / f'checkpoint_epoch_{epoch+1}.pth')
            self._cleanup_old_checkpoints()
        
        # Save history
        self.history.append({'epoch': epoch, **metrics})
        with open(self.save_dir / 'history.json', 'w') as f:
            json.dump(self.history, f, indent=2)
    
    def load(self, model, optimizer=None, scheduler=None, scaler=None,
             checkpoint_path=None, load_best: bool = False) -> int:
        """Load checkpoint. Returns start epoch."""
        if checkpoint_path is None:
            if load_best:
                checkpoint_path = self.load_dir / 'best_model.pth'
            else:
                checkpoint_path = self.load_dir / 'last_checkpoint.pth'
        
        checkpoint_path = Path(checkpoint_path)
        if not checkpoint_path.exists():
            print("No checkpoint found, starting fresh training")
            return 0
        
        print(f"Loading checkpoint from {checkpoint_path}")
        ckpt = torch.load(checkpoint_path, map_location=cfg.DEVICE, weights_only=False)
        
        # Load model (handle torch.compile / DDP prefixes)
        raw_sd = ckpt.get('model_state_dict', ckpt)
        cleaned_sd = {}
        for k, v in raw_sd.items():
            k2 = k.replace('module.', '').replace('_orig_mod.', '')
            cleaned_sd[k2] = v

        target_model = model._orig_mod if hasattr(model, '_orig_mod') else model
        missing, unexpected = target_model.load_state_dict(cleaned_sd, strict=False)
        if missing or unexpected:
            print(f"  Load keys: missing={len(missing)} unexpected={len(unexpected)}")
        
        # Load optimizer
        if optimizer and 'optimizer_state_dict' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        
        # Load scheduler
        if scheduler and ckpt.get('scheduler_state_dict'):
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        
        # Load scaler
        if scaler and ckpt.get('scaler_state_dict'):
            scaler.load_state_dict(ckpt['scaler_state_dict'])
        
        # Restore best tracking
        self.best_score = ckpt.get('best_score', -1)
        self.best_epoch = ckpt.get('best_epoch', -1)
        
        # Load history
        history_path = self.save_dir / 'history.json'
        if history_path.exists():
            with open(history_path, 'r') as f:
                self.history = json.load(f)
        
        start_epoch = ckpt['epoch'] + 1
        print(f"Resumed from epoch {ckpt['epoch']}")
        if self.best_score > 0:
            print(f"Best score so far: {self.best_score:.4f} at epoch {self.best_epoch}")
        
        return start_epoch
    
    def _cleanup_old_checkpoints(self):
        """Remove old periodic checkpoints, keeping only max_keep."""
        checkpoints = sorted(self.save_dir.glob('checkpoint_epoch_*.pth'))
        while len(checkpoints) > self.max_keep:
            checkpoints[0].unlink()
            checkpoints = checkpoints[1:]

print("CheckpointManager defined")
print("  - Save to: cfg.CHECKPOINT_DIR")
print("  - Load from: separate load_dir (can be Kaggle input dataset)")

In [ ]:
# =============================================================================
# CELL 3: HIGH-PERFORMANCE DATASET (Preload to RAM)
# =============================================================================

def load_volume_fast(path: Path) -> np.ndarray:
    """Load 3D TIF volume."""
    try:
        import tifffile
        return tifffile.imread(str(path))
    except ImportError:
        im = Image.open(path)
        slices = [np.array(page) for page in ImageSequence.Iterator(im)]
        return np.stack(slices, axis=0)


def compute_signed_distance_map(label: np.ndarray) -> np.ndarray:
    """
    Compute signed distance transform for boundary loss.
    Negative inside foreground, positive outside.
    Normalized by max distance to keep values in a reasonable range.
    """
    fg = label > 0
    if fg.sum() == 0 or fg.sum() == fg.size:
        return np.zeros_like(label, dtype=np.float32)
    
    # Distance from foreground boundary (outside -> positive)
    dist_outside = distance_transform_edt(~fg).astype(np.float32)
    # Distance from background boundary (inside -> negative)
    dist_inside = distance_transform_edt(fg).astype(np.float32)
    
    # Signed distance: positive outside, negative inside
    signed_dist = dist_outside - dist_inside
    
    # Normalize to [-1, 1] range for stable training
    max_dist = max(np.abs(signed_dist).max(), 1.0)
    signed_dist = signed_dist / max_dist
    
    return signed_dist


class VesuviusDatasetFast(Dataset):
    """
    High-performance dataset - PRELOADS ALL DATA TO RAM.
    Designed for 220GB RAM systems.
    
    NO disk I/O during training = maximum GPU utilization.
    
    When finetune_mode=True, also returns precomputed distance maps
    for boundary loss computation.
    """
    
    def __init__(
        self,
        csv_path: Path,
        images_dir: Path,
        labels_dir: Path,
        patch_size: Tuple[int, int, int] = (192, 192, 192),
        is_train: bool = True,
        data_fraction: float = 1.0,
        patches_per_volume: int = 8,
        volume_ids: Optional[List[str]] = None,
        preload: bool = True,
        finetune_mode: bool = False,
    ):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.patch_size = patch_size
        self.is_train = is_train
        self.patches_per_volume = patches_per_volume
        self.finetune_mode = finetune_mode
        
        # Load CSV and filter
        df = pd.read_csv(csv_path)
        valid_ids = []
        for idx in df['id'].values:
            if (self.images_dir / f"{idx}.tif").exists() and \
               (self.labels_dir / f"{idx}.tif").exists():
                valid_ids.append(idx)

        if volume_ids is not None:
            allowed = set([str(v) for v in volume_ids])
            valid_ids = [vid for vid in valid_ids if str(vid) in allowed]
        
        if data_fraction < 1.0:
            n = max(1, int(len(valid_ids) * data_fraction))
            random.shuffle(valid_ids)
            valid_ids = valid_ids[:n]
        
        self.volume_ids = valid_ids
        
        # PRELOAD ALL DATA TO RAM
        self.volumes = {}
        self.fg_coords = {}
        
        if preload:
            print(f"Preloading {len(self.volume_ids)} volumes to RAM...")
            for vol_id in tqdm(self.volume_ids, desc="Loading"):
                img = load_volume_fast(self.images_dir / f"{vol_id}.tif").astype(np.float32)
                lbl = load_volume_fast(self.labels_dir / f"{vol_id}.tif").astype(np.uint8)
                
                # Normalize
                img = (img - img.mean()) / (img.std() + 1e-8)
                
                self.volumes[vol_id] = (img, lbl)
                
                # Cache foreground coords
                fg = np.argwhere(lbl == 1)
                if len(fg) > 10000:
                    fg = fg[np.random.choice(len(fg), 10000, replace=False)]
                self.fg_coords[vol_id] = fg if len(fg) > 0 else None
            
            # Estimate memory usage
            sample_vol = next(iter(self.volumes.values()))
            vol_size_mb = (sample_vol[0].nbytes + sample_vol[1].nbytes) / 1e6
            total_gb = vol_size_mb * len(self.volumes) / 1000
            print(f"Loaded {len(self.volumes)} volumes ({total_gb:.1f} GB in RAM)")
        
        print(f"Dataset ready: {len(self)} samples" + 
              (" [FINETUNE MODE: will compute distance maps on-the-fly]" if finetune_mode else ""))
    
    def __len__(self):
        return len(self.volume_ids) * self.patches_per_volume
    
    def __getitem__(self, idx):
        vol_idx = idx // self.patches_per_volume
        vol_id = self.volume_ids[vol_idx]
        
        image, label = self.volumes[vol_id]
        
        # Extract patch
        d, h, w = image.shape
        pd, ph, pw = self.patch_size
        
        # Pad if needed
        if d < pd or h < ph or w < pw:
            pad_d, pad_h, pad_w = max(0, pd-d), max(0, ph-h), max(0, pw-w)
            image = np.pad(image, ((0,pad_d),(0,pad_h),(0,pad_w)), mode='reflect')
            label = np.pad(label, ((0,pad_d),(0,pad_h),(0,pad_w)), mode='constant', constant_values=2)
            d, h, w = image.shape
        
        # Foreground-centered sampling (60% of time)
        fg = self.fg_coords.get(vol_id)
        if self.is_train and random.random() < 0.6 and fg is not None and len(fg) > 0:
            c = fg[random.randint(0, len(fg)-1)]
            z = max(0, min(c[0] - pd//2, d - pd))
            y = max(0, min(c[1] - ph//2, h - ph))
            x = max(0, min(c[2] - pw//2, w - pw))
        else:
            z = random.randint(0, max(0, d - pd))
            y = random.randint(0, max(0, h - ph))
            x = random.randint(0, max(0, w - pw))
        
        img_patch = image[z:z+pd, y:y+ph, x:x+pw].copy()
        lbl_patch = label[z:z+pd, y:y+ph, x:x+pw].copy()
        
        # Augmentation
        if self.is_train:
            # Flips
            for ax in range(3):
                if random.random() > 0.5:
                    img_patch = np.flip(img_patch, ax)
                    lbl_patch = np.flip(lbl_patch, ax)
            
            # Rotation
            k = random.randint(0, 3)
            if k:
                img_patch = np.rot90(img_patch, k, (1,2))
                lbl_patch = np.rot90(lbl_patch, k, (1,2))
            
            # Make contiguous after transforms
            img_patch = np.ascontiguousarray(img_patch)
            lbl_patch = np.ascontiguousarray(lbl_patch)
            
            # Intensity
            if random.random() > 0.5:
                img_patch = img_patch * random.uniform(0.8, 1.2)
            if random.random() > 0.5:
                img_patch = img_patch + random.uniform(-0.1, 0.1)

            # Gamma correction on normalized data (preserve sign)
            if random.random() > 0.5:
                gamma = random.uniform(0.7, 1.5)
                img_patch = np.sign(img_patch) * (np.abs(img_patch) ** gamma)
            # Gaussian noise
            if random.random() > 0.5:
                sigma = random.uniform(0.01, 0.03)
                img_patch = img_patch + np.random.normal(0, sigma, size=img_patch.shape).astype(img_patch.dtype)
        
        fg_mask = (lbl_patch == 1).astype(np.float32)
        ig_mask = (lbl_patch == 2).astype(np.float32)
        
        result = {
            'image': torch.from_numpy(img_patch).unsqueeze(0).float(),
            'label': torch.from_numpy(fg_mask).unsqueeze(0).float(),
            'ignore_mask': torch.from_numpy(ig_mask).unsqueeze(0).float(),
        }
        
        # Compute distance map for boundary loss (on-the-fly per patch)
        if self.finetune_mode:
            dist_map = compute_signed_distance_map(fg_mask)
            result['dist_map'] = torch.from_numpy(dist_map).unsqueeze(0).float()
        
        return result


class PrefetchLoader:
    """GPU prefetching - overlaps data transfer with compute."""
    def __init__(self, loader, device):
        self.loader = loader
        self.device = device
        self.stream = torch.cuda.Stream()
    
    def __iter__(self):
        first = True
        next_batch = None
        
        for batch in self.loader:
            if first:
                # First batch - transfer synchronously
                batch = {k: v.to(self.device, non_blocking=True) if isinstance(v, torch.Tensor) else v 
                         for k, v in batch.items()}
                first = False
            else:
                # Wait for previous prefetch, swap batches
                torch.cuda.current_stream().wait_stream(self.stream)
                batch, next_batch = next_batch, None
            
            # Start prefetching next batch
            try:
                next_data = next(self._iter)
                with torch.cuda.stream(self.stream):
                    next_batch = {k: v.to(self.device, non_blocking=True) if isinstance(v, torch.Tensor) else v 
                                  for k, v in next_data.items()}
            except StopIteration:
                pass
            
            yield batch
        
        # Yield last prefetched batch
        if next_batch is not None:
            torch.cuda.current_stream().wait_stream(self.stream)
            yield next_batch
    
    def __len__(self):
        return len(self.loader)
    
    def _create_iter(self):
        self._iter = iter(self.loader)


# Use standard loader with pin_memory - simpler and works well
print("High-performance dataset ready")
print("All data will be preloaded to RAM - zero disk I/O during training")
if cfg.FINETUNE_MODE:
    print("FINETUNE_MODE: Distance maps will be computed on-the-fly per patch")

In [5]:
# =============================================================================
# CELL 4: MODEL (6-stage ResEncUNet3D + scSE + GroupNorm)
# =============================================================================

class ConvBlock(nn.Module):
    """3D convolution block with GroupNorm (better for small batch sizes)."""
    def __init__(self, in_ch, out_ch, num_groups=8):
        super().__init__()
        # Dynamic group selection: use 8 groups, but ensure divisibility
        num_groups = min(num_groups, out_ch)
        while out_ch % num_groups != 0:
            num_groups -= 1
        
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(num_groups, out_ch),  # GroupNorm instead of InstanceNorm
            nn.LeakyReLU(0.01, inplace=True),
        )
    
    def forward(self, x):
        return self.conv(x)


class ResBlock(nn.Module):
    """Residual block with N convolutions."""
    def __init__(self, channels, n_convs=2):
        super().__init__()
        self.blocks = nn.Sequential(
            *[ConvBlock(channels, channels) for _ in range(n_convs)]
        )
    
    def forward(self, x):
        return x + self.blocks(x)


class scSEBlock(nn.Module):
    """Concurrent Spatial and Channel Squeeze-and-Excitation."""
    def __init__(self, channels, reduction=2):
        super().__init__()
        self.cse_pool = nn.AdaptiveAvgPool3d(1)
        self.cse_fc1 = nn.Linear(channels, channels // reduction)
        self.cse_fc2 = nn.Linear(channels // reduction, channels)
        self.sse_conv = nn.Conv3d(channels, 1, 1)
    
    def forward(self, x):
        b, c, d, h, w = x.shape
        cse = self.cse_pool(x).view(b, c)
        cse = F.relu(self.cse_fc1(cse))
        cse = torch.sigmoid(self.cse_fc2(cse)).view(b, c, 1, 1, 1)
        sse = torch.sigmoid(self.sse_conv(x))
        return x * cse + x * sse


class ResEncUNet3D(nn.Module):
    """
    6-stage Residual Encoder U-Net with scSE attention and GroupNorm.
    
    GroupNorm advantages for 3D segmentation:
    - Works well with small batch sizes (1-4)
    - More stable gradients than InstanceNorm
    - Better feature preservation for thin structures
    """
    
    def __init__(
        self,
        in_ch: int = 1,
        out_ch: int = 1,
        features: List[int] = None,
        n_blocks: List[int] = None,
        use_scse: bool = True,
        use_deep_supervision: bool = True,
    ):
        super().__init__()
        
        if features is None:
            features = [32, 64, 128, 256, 320, 320]
        if n_blocks is None:
            n_blocks = [1, 3, 4, 6, 6, 6]
        
        self.features = features
        self.use_scse = use_scse
        self.use_deep_supervision = use_deep_supervision
        self.n_stages = len(features)
        
        # Encoder
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i - 1]
            encoder = nn.Sequential(
                ConvBlock(in_channels, feat),
                *[ResBlock(feat, n_convs=2) for _ in range(nb)]
            )
            self.encoders.append(encoder)
            
            if use_scse:
                self.attentions.append(scSEBlock(feat))
            else:
                self.attentions.append(nn.Identity())
            
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, kernel_size=2, stride=2))
        
        # Decoder
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        
        for i in range(len(features) - 2, -1, -1):
            up_feat = features[i + 1]
            out_feat = features[i]
            self.ups.append(nn.ConvTranspose3d(up_feat, out_feat, kernel_size=2, stride=2))
            self.dec_convs.append(ConvBlock(out_feat * 2, out_feat))
        
        self.final = nn.Conv3d(features[0], out_ch, 1)
        
        # Deep supervision heads
        if use_deep_supervision:
            self.ds_heads = nn.ModuleList()
            n_ds_outputs = min(3, len(features) - 1)
            for i in range(n_ds_outputs):
                ds_in_channels = features[-(i + 2)]
                self.ds_heads.append(nn.Conv3d(ds_in_channels, out_ch, 1))
    
    def forward(self, x):
        enc_features = []
        
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = enc(x)
            x = att(x)
            enc_features.append(x)
            if i < len(self.pools):
                x = self.pools[i](x)
        
        enc_features = enc_features[::-1]
        x = enc_features[0]
        
        ds_outputs = []
        
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i + 1]
            
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
            
            if self.use_deep_supervision and self.training and i < len(self.ds_heads):
                ds_outputs.append(self.ds_heads[i](x))
        
        out = self.final(x)
        
        if self.use_deep_supervision and self.training:
            return {'output': out, 'deep': ds_outputs}
        return out


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model with GroupNorm defined")
print("GroupNorm advantages:")
print("  - Better for batch size 1-4")
print("  - Stable gradients")
print("  - Better thin structure preservation")

Model with GroupNorm defined
GroupNorm advantages:
  - Better for batch size 1-4
  - Stable gradients
  - Better thin structure preservation


In [ ]:
# =============================================================================
# CELL 5: LOSS FUNCTIONS
# =============================================================================
# Original: Dice + BCE + SkeletonRecall + soft-clDice
# Fine-tuning adds: BoundaryLoss + HausdorffDTLoss
# =============================================================================

class DiceLoss(nn.Module):
    """Dice loss with ignore mask support."""
    def __init__(self, smooth: float = 1e-5):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target, mask=None):
        pred = torch.sigmoid(pred)
        
        if mask is not None:
            pred = pred * (1 - mask)
            target = target * (1 - mask)
        
        intersection = (pred * target).sum()
        union = pred.sum() + target.sum()
        
        dice = (2 * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice


class BCEWithMask(nn.Module):
    """BCE loss with ignore mask support."""
    def forward(self, pred, target, mask=None):
        if mask is not None:
            valid = 1 - mask
            loss = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
            loss = (loss * valid).sum() / (valid.sum() + 1e-8)
        else:
            loss = F.binary_cross_entropy_with_logits(pred, target)
        return loss


def soft_skeletonize(x, num_iter=40):
    """
    Soft skeletonization using iterative min-max pooling.
    From clDice paper (CVPR 2021).
    """
    for _ in range(num_iter):
        min_pool = -F.max_pool3d(-x, 3, stride=1, padding=1)
        max_min_pool = F.max_pool3d(min_pool, 3, stride=1, padding=1)
        x = F.relu(x - max_min_pool)
    return x


class SoftClDiceLoss(nn.Module):
    """
    Soft clDice (centerline Dice) loss.
    Preserves topology by focusing on skeleton connectivity.
    """
    def __init__(self, smooth: float = 1e-5, num_iter: int = 10):
        super().__init__()
        self.smooth = smooth
        self.num_iter = num_iter
    
    def forward(self, pred, target, mask=None):
        pred = torch.sigmoid(pred)
        
        if mask is not None:
            pred = pred * (1 - mask)
            target = target * (1 - mask)
        
        skel_pred = soft_skeletonize(pred, self.num_iter)
        skel_target = soft_skeletonize(target, self.num_iter)
        
        tprec = ((skel_pred * target).sum() + self.smooth) / (skel_pred.sum() + self.smooth)
        tsens = ((skel_target * pred).sum() + self.smooth) / (skel_target.sum() + self.smooth)
        cldice = 2 * tprec * tsens / (tprec + tsens + self.smooth)
        
        return 1 - cldice


class SkeletonRecallLoss(nn.Module):
    """Skeleton Recall Loss."""
    def __init__(self, smooth: float = 1e-5, tube_radius: int = 2):
        super().__init__()
        self.smooth = smooth
        self.tube_radius = tube_radius
    
    def forward(self, pred, target, mask=None):
        pred_sigmoid = torch.sigmoid(pred)
        
        if mask is not None:
            pred_sigmoid = pred_sigmoid * (1 - mask)
            target = target * (1 - mask)
        
        target_skel = soft_skeletonize(target, num_iter=10)
        
        if self.tube_radius > 0:
            target_tube = F.max_pool3d(
                target_skel,
                kernel_size=2 * self.tube_radius + 1,
                stride=1,
                padding=self.tube_radius
            )
        else:
            target_tube = target_skel
        
        recall = (pred_sigmoid * target_tube).sum() / (target_tube.sum() + self.smooth)
        return 1 - recall


# =============================================================================
# NEW FINE-TUNING LOSSES
# =============================================================================

class BoundaryLoss(nn.Module):
    """
    Boundary Loss (Kervadec et al., MIDL 2019).
    
    Uses precomputed signed distance transforms of the ground truth boundary.
    Penalizes predictions that are far from the true boundary.
    This is a regional loss that doesn't need thresholding - it directly
    uses the soft probability output.
    
    The key insight: multiply soft predictions by the distance map.
    Voxels far from the boundary get a large penalty, pushing the model
    to be precise at boundaries.
    """
    def __init__(self):
        super().__init__()
    
    def forward(self, pred, dist_map, mask=None):
        """
        Args:
            pred: logits (B, 1, D, H, W)
            dist_map: signed distance transform (B, 1, D, H, W)
                      negative inside, positive outside the target
            mask: ignore mask (B, 1, D, H, W)
        """
        pred_soft = torch.sigmoid(pred)
        
        if mask is not None:
            valid = 1 - mask
            # Multiply predictions by distance map, masked
            loss = (pred_soft * dist_map * valid).sum() / (valid.sum() + 1e-8)
        else:
            loss = (pred_soft * dist_map).mean()
        
        return loss


class HausdorffDTLoss(nn.Module):
    """
    Hausdorff Distance Transform Loss (Karimi & Salcudean, 2019).
    
    Approximates Hausdorff distance using distance transforms.
    Computed on-the-fly from the soft predictions and targets.
    
    This loss penalizes large local deviations between predicted and
    target boundaries, which is exactly what the competition metric cares about.
    
    Uses alpha parameter to control sensitivity:
    - alpha=1: linear penalty (like boundary loss)
    - alpha=2: squared penalty (emphasizes large errors more)
    """
    def __init__(self, alpha: float = 2.0):
        super().__init__()
        self.alpha = alpha
    
    def forward(self, pred, target, mask=None):
        """
        Args:
            pred: logits (B, 1, D, H, W)
            target: binary target (B, 1, D, H, W)
            mask: ignore mask (B, 1, D, H, W)
        """
        pred_soft = torch.sigmoid(pred)
        
        if mask is not None:
            pred_soft = pred_soft * (1 - mask)
            target = target * (1 - mask)
        
        # Compute distance transforms on CPU (scipy), then move back
        # This is done per-sample in the batch
        batch_size = pred.shape[0]
        total_loss = torch.tensor(0.0, device=pred.device)
        
        with torch.no_grad():
            for b in range(batch_size):
                target_np = target[b, 0].cpu().numpy()
                pred_np = (pred_soft[b, 0] > 0.5).cpu().numpy()
                
                # Distance transform of target boundary
                if target_np.sum() > 0 and (1 - target_np).sum() > 0:
                    dt_target = distance_transform_edt(1 - target_np) + distance_transform_edt(target_np)
                else:
                    dt_target = np.zeros_like(target_np)
                
                # Distance transform of prediction boundary
                if pred_np.sum() > 0 and (1 - pred_np).sum() > 0:
                    dt_pred = distance_transform_edt(1 - pred_np) + distance_transform_edt(pred_np)
                else:
                    dt_pred = np.zeros_like(pred_np)
                
                dt_target_t = torch.from_numpy(dt_target.astype(np.float32)).to(pred.device)
                dt_pred_t = torch.from_numpy(dt_pred.astype(np.float32)).to(pred.device)
                
                # Store for gradient computation
                self._dt_target = dt_target_t
                self._dt_pred = dt_pred_t
        
        # Now compute loss WITH gradients through pred_soft
        for b in range(batch_size):
            target_np = target[b, 0].detach().cpu().numpy()
            pred_soft_b = pred_soft[b, 0]
            target_b = target[b, 0]
            
            # Recompute distance transforms (no grad needed for these)
            with torch.no_grad():
                if target_np.sum() > 0 and (1 - target_np).sum() > 0:
                    dt_target = distance_transform_edt(1 - target_np)
                else:
                    dt_target = np.zeros_like(target_np)
                
                pred_np = (pred_soft_b > 0.5).cpu().numpy()
                if pred_np.sum() > 0 and (1 - pred_np).sum() > 0:
                    dt_pred = distance_transform_edt(1 - pred_np)
                else:
                    dt_pred = np.zeros_like(pred_np)
            
            dt_target_t = torch.from_numpy(dt_target.astype(np.float32)).to(pred.device)
            dt_pred_t = torch.from_numpy(dt_pred.astype(np.float32)).to(pred.device)
            
            # Loss: weighted sum of (pred - target) errors at boundary regions
            # pred_soft * dt_target^alpha penalizes FP far from target boundary
            # (1 - pred_soft) * dt_pred^alpha penalizes FN far from pred boundary
            loss_b = (
                (pred_soft_b * dt_target_t ** self.alpha).mean() +
                ((1 - pred_soft_b) * target_b * dt_pred_t ** self.alpha).mean()
            )
            total_loss = total_loss + loss_b
        
        return total_loss / batch_size


# =============================================================================
# COMBINED LOSS (supports both normal and fine-tuning modes)
# =============================================================================

class CombinedLoss(nn.Module):
    """Combined loss with scheduling for topology losses.
    
    When finetune_mode=True, also includes BoundaryLoss and HausdorffDTLoss
    with a warmup schedule to gradually introduce them.
    """
    def __init__(
        self,
        dice_weight: float = 0.3,
        bce_weight: float = 0.2,
        skeleton_weight: float = 0.25,
        cldice_weight: float = 0.25,
        skeleton_start: int = 300,
        skeleton_warmup: int = 300,
        cldice_start: int = 600,
        cldice_warmup: int = 300,
        ds_weights: List[float] = None,
        # Fine-tuning parameters
        finetune_mode: bool = False,
        boundary_weight: float = 0.15,
        hausdorff_weight: float = 0.10,
        boundary_warmup_epochs: int = 30,
    ):
        super().__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.skeleton_weight = skeleton_weight
        self.cldice_weight = cldice_weight
        
        self.skeleton_start = skeleton_start
        self.skeleton_warmup = skeleton_warmup
        self.cldice_start = cldice_start
        self.cldice_warmup = cldice_warmup
        
        if ds_weights is None:
            ds_weights = [0.5, 0.25, 0.125]
        self.ds_weights = ds_weights
        
        # Standard losses
        self.dice_loss = DiceLoss()
        self.bce_loss = BCEWithMask()
        self.skeleton_loss = SkeletonRecallLoss()
        self.cldice_loss = SoftClDiceLoss()
        
        # Fine-tuning losses
        self.finetune_mode = finetune_mode
        self.boundary_weight = boundary_weight
        self.hausdorff_weight = hausdorff_weight
        self.boundary_warmup_epochs = boundary_warmup_epochs
        self.finetune_start_epoch = None  # Set when training starts
        
        if finetune_mode:
            self.boundary_loss = BoundaryLoss()
            self.hausdorff_loss = HausdorffDTLoss(alpha=2.0)
    
    def _get_scale(self, epoch, start, warmup):
        if epoch < start:
            return 0.0
        elif epoch < start + warmup:
            return (epoch - start) / warmup
        return 1.0
    
    def _get_boundary_scale(self, epoch):
        """Get boundary loss scale with warmup from fine-tune start."""
        if self.finetune_start_epoch is None:
            self.finetune_start_epoch = epoch
        
        elapsed = epoch - self.finetune_start_epoch
        if elapsed < 0:
            return 0.0
        elif elapsed < self.boundary_warmup_epochs:
            return elapsed / self.boundary_warmup_epochs
        return 1.0
    
    def forward(self, output, target, ignore_mask, epoch, dist_map=None):
        if isinstance(output, dict):
            pred = output['output']
            deep_outputs = output.get('deep', [])
        else:
            pred = output
            deep_outputs = []
        
        skel_scale = self._get_scale(epoch, self.skeleton_start, self.skeleton_warmup)
        cldice_scale = self._get_scale(epoch, self.cldice_start, self.cldice_warmup)
        
        # Base losses
        dice = self.dice_loss(pred, target, ignore_mask)
        bce = self.bce_loss(pred, target, ignore_mask)
        
        if skel_scale > 0:
            skel = self.skeleton_loss(pred, target, ignore_mask)
        else:
            skel = torch.tensor(0.0, device=pred.device)
        
        if cldice_scale > 0:
            cldice = self.cldice_loss(pred, target, ignore_mask)
        else:
            cldice = torch.tensor(0.0, device=pred.device)
        
        total = (
            self.dice_weight * dice +
            self.bce_weight * bce +
            self.skeleton_weight * skel_scale * skel +
            self.cldice_weight * cldice_scale * cldice
        )
        
        # Fine-tuning losses
        boundary_val = 0.0
        hausdorff_val = 0.0
        boundary_scale = 0.0
        
        if self.finetune_mode:
            boundary_scale = self._get_boundary_scale(epoch)
            
            if boundary_scale > 0:
                # Boundary loss (needs precomputed distance map)
                if dist_map is not None:
                    b_loss = self.boundary_loss(pred, dist_map, ignore_mask)
                    total = total + self.boundary_weight * boundary_scale * b_loss
                    boundary_val = b_loss.item()
                
                # Hausdorff DT loss (computed on-the-fly)
                h_loss = self.hausdorff_loss(pred, target, ignore_mask)
                total = total + self.hausdorff_weight * boundary_scale * h_loss
                hausdorff_val = h_loss.item()
        
        # Deep supervision
        for i, ds_out in enumerate(deep_outputs):
            if i >= len(self.ds_weights):
                break
            ds_target = F.interpolate(target, size=ds_out.shape[2:], mode='nearest')
            ds_mask = F.interpolate(ignore_mask, size=ds_out.shape[2:], mode='nearest')
            ds_dice = self.dice_loss(ds_out, ds_target, ds_mask)
            total = total + self.ds_weights[i] * ds_dice
        
        return {
            'total': total,
            'dice': dice.item(),
            'bce': bce.item(),
            'skeleton': skel.item() if skel_scale > 0 else 0.0,
            'cldice': cldice.item() if cldice_scale > 0 else 0.0,
            'skel_scale': skel_scale,
            'cldice_scale': cldice_scale,
            'boundary': boundary_val,
            'hausdorff': hausdorff_val,
            'boundary_scale': boundary_scale,
        }

print("Loss functions defined")
if cfg.FINETUNE_MODE:
    print("FINE-TUNING MODE: BoundaryLoss + HausdorffDTLoss enabled")
    print(f"  Boundary warmup: {cfg.BOUNDARY_WARMUP_EPOCHS} epochs")
    print(f"  Weights: dice={cfg.FT_DICE_WEIGHT}, bce={cfg.FT_BCE_WEIGHT}, "
          f"skel={cfg.FT_SKELETON_WEIGHT}, cldice={cfg.FT_CLDICE_WEIGHT}, "
          f"boundary={cfg.BOUNDARY_WEIGHT}, hausdorff={cfg.HAUSDORFF_WEIGHT}")
else:
    print(f"Loss scheduling:")
    print(f"  Epochs 0-{cfg.SKELETON_START_EPOCH}: Dice + BCE only")
    print(f"  Epochs {cfg.SKELETON_START_EPOCH}-{cfg.CLDICE_START_EPOCH}: Add SkeletonRecall")
    print(f"  Epochs {cfg.CLDICE_START_EPOCH}-{cfg.EPOCHS}: Add soft-clDice")

In [ ]:
# =============================================================================
# CELL 6: TRAINING FUNCTIONS
# =============================================================================

import sys

def train_one_epoch(model, loader, criterion, optimizer, scaler, device, epoch):
    """Train one epoch. Supports fine-tuning mode with boundary losses."""
    model.train()
    
    total_loss = 0
    total_dice = 0
    total_boundary = 0
    total_hausdorff = 0
    n_batches = 0
    
    pbar = tqdm(total=len(loader), desc=f"Epoch {epoch+1}", 
                file=sys.stdout, dynamic_ncols=True)
    
    for batch in loader:
        images = batch['image'].to(device, non_blocking=True)
        labels = batch['label'].to(device, non_blocking=True)
        ignore = batch['ignore_mask'].to(device, non_blocking=True)
        
        # Get distance map if available (fine-tuning mode)
        dist_map = None
        if 'dist_map' in batch:
            dist_map = batch['dist_map'].to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        with autocast(enabled=cfg.USE_AMP):
            output = model(images)
            losses = criterion(output, labels, ignore, epoch, dist_map=dist_map)
        
        scaler.scale(losses['total']).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRADIENT_CLIP)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += losses['total'].item()
        total_dice += losses['dice']
        total_boundary += losses.get('boundary', 0)
        total_hausdorff += losses.get('hausdorff', 0)
        n_batches += 1
        
        # Build postfix dict
        postfix = {
            'loss': f"{losses['total'].item():.3f}",
            'dice': f"{losses['dice']:.3f}",
        }
        if cfg.FINETUNE_MODE and losses.get('boundary_scale', 0) > 0:
            postfix['bnd'] = f"{losses.get('boundary', 0):.3f}"
            postfix['haus'] = f"{losses.get('hausdorff', 0):.3f}"
        
        pbar.update(1)
        pbar.set_postfix(postfix)
        
        if n_batches % 50 == 0:
            sys.stdout.flush()
    
    pbar.close()
    sys.stdout.flush()
    
    metrics = {
        'train_loss': total_loss / n_batches,
        'train_dice_loss': total_dice / n_batches,
    }
    
    if cfg.FINETUNE_MODE:
        metrics['train_boundary'] = total_boundary / n_batches
        metrics['train_hausdorff'] = total_hausdorff / n_batches
    
    return metrics


@torch.no_grad()
def validate(model, loader, device):
    """Fast validation."""
    model.eval()
    
    total_dice = 0
    n_samples = 0
    
    for batch in tqdm(loader, desc="Val", file=sys.stdout, leave=False):
        images = batch['image'].to(device, non_blocking=True)
        labels = batch['label'].numpy()
        ignore = batch['ignore_mask'].numpy()
        
        with autocast(enabled=cfg.USE_AMP):
            output = model(images)
            if isinstance(output, dict):
                output = output['output']
            probs = torch.sigmoid(output).cpu().numpy()
        
        for b in range(images.shape[0]):
            pred = (probs[b, 0] > 0.5).astype(np.uint8)
            tgt = labels[b, 0].astype(np.uint8)
            ign = ignore[b, 0] > 0.5
            pred[ign] = 0
            tgt[ign] = 0
            
            inter = (pred & tgt).sum()
            union = pred.sum() + tgt.sum()
            total_dice += (2 * inter + 1e-5) / (union + 1e-5)
            n_samples += 1
    
    sys.stdout.flush()
    return {'val_dice': total_dice / max(n_samples, 1)}



@torch.no_grad()
def validate_full_volume(model, val_dataset, device, threshold=0.5, overlap=0.5, patch_size=(192,192,192)):
    """Full-volume validation using sliding window inference."""
    model.eval()
    scores = []
    for vol_id in val_dataset.volume_ids:
        img, lbl = val_dataset.volumes[vol_id]
        prob = sliding_window_inference(model, img, patch_size, overlap, device)
        pred = (prob > threshold).astype(np.uint8)
        tgt = (lbl == 1).astype(np.uint8)
        ign = (lbl == 2)
        pred[ign] = 0
        tgt[ign] = 0
        inter = (pred & tgt).sum()
        union = pred.sum() + tgt.sum()
        dice = (2 * inter + 1e-5) / (union + 1e-5)
        scores.append(dice)
        print(f"  [VAL] {vol_id}: dice={dice:.4f}")
    mean_dice = float(np.mean(scores)) if scores else 0.0
    return {'val_dice': mean_dice}

print("Training functions ready")
if cfg.FINETUNE_MODE:
    print("  Fine-tuning mode: boundary & hausdorff losses tracked")

In [ ]:
# =============================================================================
# CELL 10: INFERENCE FUNCTIONS
# =============================================================================

@torch.no_grad()
def sliding_window_inference(
    model,
    volume: np.ndarray,
    patch_size: Tuple[int, int, int] = (192, 192, 192),
    overlap: float = 0.5,
    device: str = 'cuda',
) -> np.ndarray:
    """
    Sliding window inference with Gaussian weighting.
    
    Args:
        model: Trained model
        volume: Input volume (D, H, W)
        patch_size: Patch size
        overlap: Overlap fraction
        device: Device for inference
    
    Returns:
        Probability map (D, H, W)
    """
    model.eval()
    
    D, H, W = volume.shape
    pd, ph, pw = patch_size
    sd, sh, sw = int(pd * (1 - overlap)), int(ph * (1 - overlap)), int(pw * (1 - overlap))
    
    # Output arrays
    pred_sum = np.zeros((D, H, W), dtype=np.float32)
    count = np.zeros((D, H, W), dtype=np.float32)
    
    # Gaussian weighting
    sigma = 0.125
    gauss_z = np.exp(-0.5 * ((np.arange(pd) - pd/2) / (pd * sigma)) ** 2)
    gauss_y = np.exp(-0.5 * ((np.arange(ph) - ph/2) / (ph * sigma)) ** 2)
    gauss_x = np.exp(-0.5 * ((np.arange(pw) - pw/2) / (pw * sigma)) ** 2)
    gauss_weight = gauss_z[:, None, None] * gauss_y[None, :, None] * gauss_x[None, None, :]
    gauss_weight = gauss_weight.astype(np.float32)
    
    # Generate positions
    z_pos = list(set(list(range(0, max(1, D-pd+1), sd)) + ([D-pd] if D > pd else [0])))
    y_pos = list(set(list(range(0, max(1, H-ph+1), sh)) + ([H-ph] if H > ph else [0])))
    x_pos = list(set(list(range(0, max(1, W-pw+1), sw)) + ([W-pw] if W > pw else [0])))
    
    # Normalize volume
    vol_norm = (volume - volume.mean()) / (volume.std() + 1e-8)
    
    for z in tqdm(z_pos, desc="Inference", leave=False):
        for y in y_pos:
            for x in x_pos:
                # Extract patch
                patch = vol_norm[z:z+pd, y:y+ph, x:x+pw].astype(np.float32)
                actual_shape = patch.shape
                
                # Pad if needed
                if patch.shape != (pd, ph, pw):
                    pad = [(0, pd-patch.shape[0]), (0, ph-patch.shape[1]), (0, pw-patch.shape[2])]
                    patch = np.pad(patch, pad, mode='reflect')
                
                # Inference
                inp = torch.from_numpy(patch[None, None]).to(device)
                with autocast(enabled=cfg.USE_AMP):
                    out = model(inp)
                    if isinstance(out, dict):
                        out = out['output']
                    out = torch.sigmoid(out).squeeze().cpu().numpy()
                
                # Crop to actual size
                out = out[:actual_shape[0], :actual_shape[1], :actual_shape[2]]
                weight = gauss_weight[:actual_shape[0], :actual_shape[1], :actual_shape[2]]
                
                # Accumulate
                pred_sum[z:z+out.shape[0], y:y+out.shape[1], x:x+out.shape[2]] += out * weight
                count[z:z+out.shape[0], y:y+out.shape[1], x:x+out.shape[2]] += weight
    
    return pred_sum / np.maximum(count, 1e-8)


def inference_with_tta(
    model,
    volume: np.ndarray,
    patch_size: Tuple[int, int, int] = (192, 192, 192),
    overlap: float = 0.5,
    device: str = 'cuda',
    tta_level: str = 'flip',
) -> np.ndarray:
    """
    Inference with test-time augmentation.
    
    Args:
        tta_level: 'none', 'flip', or 'full'
    """
    preds = []
    
    # Original
    pred = sliding_window_inference(model, volume, patch_size, overlap, device)
    preds.append(pred)
    
    if tta_level in ['flip', 'full']:
        # Flip augmentations
        for axis in [0, 1, 2]:
            vol_flip = np.flip(volume, axis).copy()
            pred_flip = sliding_window_inference(model, vol_flip, patch_size, overlap, device)
            preds.append(np.flip(pred_flip, axis))
    
    if tta_level == 'full':
        # 90° rotation
        vol_rot = np.rot90(volume, k=1, axes=(1, 2)).copy()
        pred_rot = sliding_window_inference(model, vol_rot, patch_size, overlap, device)
        preds.append(np.rot90(pred_rot, k=-1, axes=(1, 2)))
    
    return np.mean(preds, axis=0)

print("Inference functions defined")

In [ ]:
# =============================================================================
# CELL 7: MAIN TRAINING LOOP
# =============================================================================

def main_training():
    """Training for H100 + 220GB RAM. Supports fine-tuning mode."""
    
    print("="*60)
    if cfg.FINETUNE_MODE:
        print("VESUVIUS V2 FINE-TUNING (Boundary + Hausdorff Losses)")
    else:
        print("VESUVIUS V2 TRAINING")
    print("="*60)
    print(f"Batch size: {cfg.BATCH_SIZE}")
    print(f"Workers: {cfg.NUM_WORKERS}")
    print("="*60)
    
    # Paths
    train_csv = cfg.DATA_ROOT / "train.csv"
    train_images = cfg.DATA_ROOT / "train_images"
    train_labels = cfg.DATA_ROOT / "train_labels"
    
    # Create datasets
    print("\n[1/4] Loading training data to RAM...")
    train_dataset = VesuviusDatasetFast(
        csv_path=train_csv,
        images_dir=train_images,
        labels_dir=train_labels,
        patch_size=cfg.PATCH_SIZE,
        is_train=True,
        data_fraction=cfg.DATA_FRACTION,
        patches_per_volume=cfg.PATCHES_PER_VOLUME,
        preload=True,
        finetune_mode=cfg.FINETUNE_MODE,
        volume_ids=cfg.TRAIN_VOLUME_IDS,
    
    )
    
    print("\n[2/4] Loading validation data to RAM...")
    val_dataset = VesuviusDatasetFast(
        csv_path=train_csv,
        images_dir=train_images,
        labels_dir=train_labels,
        patch_size=cfg.PATCH_SIZE,
        is_train=False,
        data_fraction=1.0,
        patches_per_volume=1,
        preload=True,
        finetune_mode=False,  # No distance maps needed for validation
        volume_ids=cfg.VAL_VOLUME_IDS,
    
    )
    
    # DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
        drop_last=True,
        persistent_workers=True,
        prefetch_factor=4,
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
    )
    
    print(f"\nTrain: {len(train_dataset)} samples, {len(train_loader)} batches/epoch")
    print(f"Val: {len(val_dataset)} samples")
    
    # Model
    print("\n[3/4] Creating model...")
    model = ResEncUNet3D(
        features=cfg.FEATURES,
        n_blocks=cfg.N_BLOCKS,
        use_scse=cfg.USE_SCSE,
        use_deep_supervision=cfg.USE_DEEP_SUPERVISION,
    ).to(cfg.DEVICE)
    
    # torch.compile for speed
    if hasattr(torch, 'compile'):
        print("Compiling model with torch.compile()...")
        model = torch.compile(model, mode='reduce-overhead')
    
    print(f"Parameters: {count_parameters(model):,}")
    
    # Loss - use fine-tuning weights if enabled
    if cfg.FINETUNE_MODE:
        criterion = CombinedLoss(
            dice_weight=cfg.FT_DICE_WEIGHT,
            bce_weight=cfg.FT_BCE_WEIGHT,
            skeleton_weight=cfg.FT_SKELETON_WEIGHT,
            cldice_weight=cfg.FT_CLDICE_WEIGHT,
            skeleton_start=cfg.SKELETON_START_EPOCH,
            skeleton_warmup=cfg.SKELETON_WARMUP_EPOCHS,
            cldice_start=cfg.CLDICE_START_EPOCH,
            cldice_warmup=cfg.CLDICE_WARMUP_EPOCHS,
            finetune_mode=True,
            boundary_weight=cfg.BOUNDARY_WEIGHT,
            hausdorff_weight=cfg.HAUSDORFF_WEIGHT,
            boundary_warmup_epochs=cfg.BOUNDARY_WARMUP_EPOCHS,
        )
        print(f"\nFine-tuning loss weights:")
        print(f"  Dice={cfg.FT_DICE_WEIGHT}, BCE={cfg.FT_BCE_WEIGHT}")
        print(f"  Skeleton={cfg.FT_SKELETON_WEIGHT}, clDice={cfg.FT_CLDICE_WEIGHT}")
        print(f"  Boundary={cfg.BOUNDARY_WEIGHT}, Hausdorff={cfg.HAUSDORFF_WEIGHT}")
        print(f"  Boundary warmup: {cfg.BOUNDARY_WARMUP_EPOCHS} epochs")
    else:
        criterion = CombinedLoss(
            dice_weight=cfg.DICE_WEIGHT,
            bce_weight=cfg.BCE_WEIGHT,
            skeleton_weight=cfg.SKELETON_WEIGHT,
            cldice_weight=cfg.CLDICE_WEIGHT,
            skeleton_start=cfg.SKELETON_START_EPOCH,
            skeleton_warmup=cfg.SKELETON_WARMUP_EPOCHS,
            cldice_start=cfg.CLDICE_START_EPOCH,
            cldice_warmup=cfg.CLDICE_WARMUP_EPOCHS,
        )
    scaler = GradScaler(enabled=cfg.USE_AMP)

    # Checkpoint manager (saves into cfg.CHECKPOINT_DIR)
    ckpt_mgr = CheckpointManager(save_dir=cfg.CHECKPOINT_DIR, max_keep=cfg.MAX_KEEP_CHECKPOINTS)

    print("\n[4/4] Initializing fine-tune / resume...")

    if cfg.FINETUNE_WEIGHTS_ONLY:
        # Load initial weights ONLY, reset optimizer/scheduler (true fine-tune)
        if cfg.INIT_CHECKPOINT_PATH is None:
            raise ValueError(
                "cfg.INIT_CHECKPOINT_PATH is None. Set it to a .pth checkpoint path (e.g. /kaggle/input/<dataset>/checkpoints/best_model.pth)."
            )
        ckpt_path = Path(cfg.INIT_CHECKPOINT_PATH)
        if not ckpt_path.exists():
            raise FileNotFoundError(f"INIT_CHECKPOINT_PATH not found: {ckpt_path}")

        print(f"Loading initial weights from: {ckpt_path}")
        ckpt = torch.load(ckpt_path, map_location=cfg.DEVICE, weights_only=False)
        state_dict = ckpt.get('model_state_dict', ckpt)

        # Clean keys (torch.compile + DDP compatibility)
        # NOTE: if model is torch.compile()'d, load into the underlying _orig_mod.
        target_model = model._orig_mod if hasattr(model, '_orig_mod') else model
        target_keys = set(target_model.state_dict().keys())

        cleaned_state = {}
        for k, v in state_dict.items():
            key = k.replace('module.', '').replace('_orig_mod.', '')
            if key in target_keys:
                cleaned_state[key] = v

        missing, unexpected = target_model.load_state_dict(cleaned_state, strict=False)
        print(f"Loaded init weights. missing={len(missing)} unexpected={len(unexpected)}")
        if len(missing) > 20:
            print("⚠️ Large number of missing keys. This usually means you loaded an incompatible checkpoint or loaded after torch.compile.")

        start_epoch = 0

        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=cfg.OVERRIDE_LR,
            momentum=cfg.MOMENTUM,
            weight_decay=cfg.WEIGHT_DECAY,
            nesterov=True,
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=cfg.EPOCHS,
            eta_min=1e-6,
        )

        print(f">>> Fine-tune schedule: {cfg.OVERRIDE_LR:.6f} -> 1e-6 over {cfg.EPOCHS} epochs")

    else:
        # Standard training / resume
        scaled_lr = cfg.LEARNING_RATE * cfg.BATCH_SIZE
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=scaled_lr,
            momentum=cfg.MOMENTUM,
            weight_decay=cfg.WEIGHT_DECAY,
            nesterov=True,
        )

        if cfg.RESUME_TRAINING:
            start_epoch = ckpt_mgr.load(model, optimizer, None, scaler)
        else:
            start_epoch = 0

        # LR override on resume
        if cfg.OVERRIDE_LR > 0 and start_epoch > 0:
            old_lr = optimizer.param_groups[0]['lr']
            for pg in optimizer.param_groups:
                pg['lr'] = cfg.OVERRIDE_LR
                pg['initial_lr'] = cfg.OVERRIDE_LR
            print(f"\n>>> LR OVERRIDE: {old_lr:.6f} -> {cfg.OVERRIDE_LR:.6f}")

            remaining_epochs = cfg.EPOCHS - start_epoch
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=remaining_epochs,
                eta_min=1e-6,
            )
            print(f">>> New cosine schedule over {remaining_epochs} epochs")
        else:
            scheduler = torch.optim.lr_scheduler.LambdaLR(
                optimizer, lambda e: (1 - e / cfg.EPOCHS) ** 0.9
            )
            if start_epoch > 0:
                for _ in range(start_epoch):
                    scheduler.step()

    # Set fine-tune start epoch for boundary loss warmup
    if cfg.FINETUNE_MODE:
        criterion.finetune_start_epoch = start_epoch
        print(f"\n>>> Boundary loss warmup: epochs {start_epoch+1} to {start_epoch + cfg.BOUNDARY_WARMUP_EPOCHS}")
    
    current_lr = optimizer.param_groups[0]['lr']
    print(f"\nStarting from epoch {start_epoch + 1}")
    print(f"Current LR: {current_lr:.6f}")
    print("="*60)
    print("TRAINING STARTED")
    print("="*60)
    
    import time
    
    for epoch in range(start_epoch, cfg.EPOCHS):
        epoch_start = time.time()
        
        train_metrics = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, cfg.DEVICE, epoch
        )
        
        scheduler.step()
        
        if (epoch + 1) % cfg.VALIDATE_EVERY == 0:
            if cfg.USE_FULL_VOLUME_VAL:
                val_metrics = validate_full_volume(model, val_dataset, cfg.DEVICE, threshold=cfg.VAL_THRESHOLD, overlap=cfg.VAL_OVERLAP, patch_size=cfg.PATCH_SIZE)
            else:
                val_metrics = validate(model, val_loader, cfg.DEVICE)
        else:
            val_metrics = {'val_dice': 0}
        
        epoch_time = time.time() - epoch_start
        
        lr = optimizer.param_groups[0]['lr']
        log_str = (f"Epoch {epoch+1}/{cfg.EPOCHS} | {epoch_time:.1f}s | LR: {lr:.6f} | "
                   f"Loss: {train_metrics['train_loss']:.4f} | "
                   f"Dice: {train_metrics['train_dice_loss']:.4f}")
        
        if cfg.FINETUNE_MODE:
            log_str += (f" | Bnd: {train_metrics.get('train_boundary', 0):.4f}"
                       f" | Haus: {train_metrics.get('train_hausdorff', 0):.4f}")
        
        if val_metrics['val_dice'] > 0:
            log_str += f" | Val: {val_metrics['val_dice']:.4f}"
        
        print(log_str)
        
        ckpt_mgr.save(model, optimizer, scheduler, scaler, epoch, 
                      {**train_metrics, **val_metrics})
    
    print("\n" + "="*60)
    print(f"DONE! Best: {ckpt_mgr.best_score:.4f} @ epoch {ckpt_mgr.best_epoch}")
    print("="*60)
    
    return model, ckpt_mgr

print("Ready to train!")
if cfg.FINETUNE_MODE:
    print("  MODE: Fine-tuning with Boundary + Hausdorff losses")
    print(f"  LR: {cfg.OVERRIDE_LR}")
    print(f"  Resume: {cfg.RESUME_TRAINING}")


In [ ]:
# =============================================================================
# CELL 8: RUN TRAINING
# =============================================================================
# IMPORTANT: This cell is the one that can accidentally overwrite cfg.EPOCHS.
# Pick ONE preset below.

RUN_PRESET = "finetune"  # "finetune" or "full_train"

if RUN_PRESET == "finetune":
    # Low-LR fine-tune from weights only (fresh optimizer/scheduler)
    cfg.FINETUNE_WEIGHTS_ONLY = True
    cfg.RESUME_TRAINING = False

    # Data: safest is to fine-tune on all training volumes (no test_id leakage)
    cfg.FINETUNE_DATA_MODE = "all"      # or "scroll"
    cfg.GLOBAL_VAL_FRACTION = 0.2       # used when FINETUNE_DATA_MODE="all"

    cfg.DATA_FRACTION = 1.0
    cfg.EPOCHS = 60
    cfg.OVERRIDE_LR = 5e-4
    cfg.SAVE_EVERY = 5
    cfg.VALIDATE_EVERY = 10

    # REQUIRED: point this at your starting checkpoint
    # Example:
    # cfg.INIT_CHECKPOINT_PATH = Path("/kaggle/input/<your_ckpt_dataset>/checkpoints/best_model.pth")

elif RUN_PRESET == "full_train":
    # Original long training recipe
    cfg.FINETUNE_WEIGHTS_ONLY = False
    cfg.RESUME_TRAINING = True
    cfg.DATA_FRACTION = 1.0
    cfg.EPOCHS = 1200
    cfg.VALIDATE_EVERY = 5

else:
    raise ValueError(f"Unknown RUN_PRESET={RUN_PRESET!r}")

model, ckpt_mgr = main_training()


VESUVIUS V2 TRAINING
Batch size: 4
Workers: 8

[1/4] Loading training data to RAM...
Preloading 786 volumes to RAM...


Loading:   0%|          | 0/786 [00:00<?, ?it/s]

Loaded 786 volumes (128.8 GB in RAM)
Dataset ready: 6288 samples

[2/4] Loading validation data to RAM...
Preloading 117 volumes to RAM...


Loading:   0%|          | 0/117 [00:00<?, ?it/s]

Loaded 117 volumes (19.2 GB in RAM)
Dataset ready: 117 samples

Train: 6288 samples, 1572 batches/epoch
Val: 117 samples

[3/4] Creating model...
Compiling model with torch.compile()...
Parameters: 111,874,554

[4/4] Checking for checkpoint...
No checkpoint found, starting fresh training

Starting from epoch 0
TRAINING STARTED


...

Epoch 47:   0%|          | 0/1572 [00:00<?, ?it/s]

Epoch 47/1200 | 637.3s | LR: 0.03859 | Loss: 0.6862 | Dice: 0.5688


Epoch 48:   0%|          | 0/1572 [00:00<?, ?it/s]

KeyboardInterrupt: 

: 

In [ ]:
# =============================================================================
# CELL 9: POST-PROCESSING FUNCTIONS
# =============================================================================

def connected_components_3d(volume, connectivity=26):
    """Compute 3D connected components."""
    if USE_CC3D:
        return cc3d.connected_components(volume, connectivity=connectivity)
    else:
        if connectivity == 26:
            struct = ndimage.generate_binary_structure(3, 3)
        elif connectivity == 6:
            struct = ndimage.generate_binary_structure(3, 1)
        else:
            struct = ndimage.generate_binary_structure(3, 2)
        labeled, _ = ndimage.label(volume, structure=struct)
        return labeled


def skeleton_guided_postprocess(
    pred_prob: np.ndarray,
    threshold: float = 0.70,
    skeleton_distance: int = 5,
    min_component_size: int = 100,
    gap_fill_threshold: float = 0.3,
) -> np.ndarray:
    """
    Skeleton-guided post-processing.
    Better than Frangi for sheet-like structures.
    
    Args:
        pred_prob: Probability map from model
        threshold: Initial binarization threshold
        skeleton_distance: Max distance from skeleton to keep
        min_component_size: Remove components smaller than this
        gap_fill_threshold: Threshold for gap filling
    
    Returns:
        Binary mask
    """
    # Step 1: Initial threshold
    binary = (pred_prob > threshold).astype(np.uint8)
    print(f"  Initial FG%: {100 * binary.mean():.2f}%")
    binary = (binary>0)
    # Step 2: Extract skeleton
    skeleton = skeletonize_3d(binary)
    print(f"  Skeleton voxels: {skeleton.sum()}")
    
    if skeleton.sum() == 0:
        print("  Warning: Empty skeleton, returning thresholded result")
        return binary
    
    # Step 3: Keep only regions connected to skeleton
    skel_dist = distance_transform_edt(~skeleton)
    skeleton_connected = binary & (skel_dist <= skeleton_distance)
    print(f"  After skeleton connection: FG%: {100 * skeleton_connected.mean():.2f}%")
    
    # Step 4: Fill small gaps along skeleton
    skel_dilated = ndimage.binary_dilation(skeleton, iterations=2)
    gap_filled = skel_dilated & (pred_prob > gap_fill_threshold)
    
    # Step 5: Combine
    final = (skeleton_connected | gap_filled).astype(np.uint8)
    
    # Step 6: Remove small components
    labeled = connected_components_3d(final)
    n_components = labeled.max()
    
    for i in range(1, n_components + 1):
        component_mask = labeled == i
        if component_mask.sum() < min_component_size:
            final[component_mask] = 0
    
    print(f"  Final FG%: {100 * final.mean():.2f}%")
    print(f"  Components: {labeled.max()} -> {connected_components_3d(final).max()}")
    
    return final


def bridge_breaking_postprocess(
    pred_prob: np.ndarray,
    threshold: float = 0.75,
    erosion_iterations: int = 2,
    min_component_size: int = 100,
) -> np.ndarray:
    """
    Alternative post-processing focused on breaking bridges.
    Good for improving VOI score.
    """
    # Step 1: Conservative threshold
    binary = (pred_prob > threshold).astype(np.uint8)
    
    # Step 2: Erosion to break thin bridges
    eroded = ndimage.binary_erosion(binary, iterations=erosion_iterations)
    
    # Step 3: Connected component filtering
    labeled = connected_components_3d(eroded.astype(np.uint8))
    
    cleaned = np.zeros_like(binary)
    for i in range(1, labeled.max() + 1):
        if (labeled == i).sum() >= min_component_size:
            cleaned[labeled == i] = 1
    
    # Step 4: Selective dilation to restore
    dilated = ndimage.binary_dilation(cleaned, iterations=erosion_iterations)
    
    # Step 5: Intersect with original to avoid over-extension
    final = (dilated & binary).astype(np.uint8)
    
    return final

print("Post-processing functions defined")

In [ ]:
# =============================================================================
# CELL 11: TEST INFERENCE ON VALIDATION DATA
# =============================================================================

def test_inference_on_sample():
    """Test inference on a single training sample."""
    
    # Load model
    model = ResEncUNet3D(
        features=cfg.FEATURES,
        n_blocks=cfg.N_BLOCKS,
        use_scse=cfg.USE_SCSE,
        use_deep_supervision=False,  # Disable for inference
    ).to(cfg.DEVICE)
    
    ckpt_mgr = CheckpointManager(cfg.CHECKPOINT_DIR)
    ckpt_mgr.load(model, load_best=True)
    model.eval()
    
    # Load a sample volume
    train_csv = pd.read_csv(cfg.DATA_ROOT / "train.csv")
    sample_id = train_csv['id'].values[0]
    
    volume = load_volume(cfg.DATA_ROOT / "train_images" / f"{sample_id}.tif")
    label = load_volume(cfg.DATA_ROOT / "train_labels" / f"{sample_id}.tif")
    
    print(f"Testing on sample: {sample_id}")
    print(f"Volume shape: {volume.shape}")
    
    # Inference
    print("\nRunning inference...")
    pred_prob = inference_with_tta(model, volume.astype(np.float32), tta_level='flip')
    
    # Post-processing comparison
    print("\n--- Post-processing comparison ---")
    
    print("\n1. Simple threshold (0.5):")
    simple = (pred_prob > 0.5).astype(np.uint8)
    print(f"   FG%: {100 * simple.mean():.2f}%")
    
    print("\n2. Conservative threshold (0.75):")
    conservative = (pred_prob > 0.75).astype(np.uint8)
    print(f"   FG%: {100 * conservative.mean():.2f}%")
    
    print("\n3. Skeleton-guided post-processing:")
    skeleton_pp = skeleton_guided_postprocess(pred_prob)
    
    print("\n4. Bridge-breaking post-processing:")
    bridge_pp = bridge_breaking_postprocess(pred_prob)
    print(f"   FG%: {100 * bridge_pp.mean():.2f}%")
    
    # Compare with ground truth
    gt = (label == 1).astype(np.uint8)
    print(f"\nGround truth FG%: {100 * gt.mean():.2f}%")
    
    # Dice scores
    def dice(pred, target):
        intersection = (pred & target).sum()
        return (2 * intersection + 1e-5) / (pred.sum() + target.sum() + 1e-5)
    
    print("\n--- Dice Scores ---")
    print(f"Simple (0.5): {dice(simple, gt):.4f}")
    print(f"Conservative (0.75): {dice(conservative, gt):.4f}")
    print(f"Skeleton-guided: {dice(skeleton_pp, gt):.4f}")
    print(f"Bridge-breaking: {dice(bridge_pp, gt):.4f}")
    
    return pred_prob, skeleton_pp

# Uncomment to run:
# pred_prob, skeleton_pp = test_inference_on_sample()

In [ ]:
# =============================================================================
# CELL 12: TRAINING STATUS CHECK
# =============================================================================

def check_training_status():
    """Check current training status and best results."""
    
    last_ckpt = cfg.CHECKPOINT_DIR / 'last_checkpoint.pth'
    best_ckpt = cfg.CHECKPOINT_DIR / 'best_model.pth'
    history_file = cfg.CHECKPOINT_DIR / 'history.json'
    
    print("="*60)
    print("TRAINING STATUS")
    print("="*60)
    
    if last_ckpt.exists():
        ckpt = torch.load(last_ckpt, map_location='cpu', weights_only=False)
        print(f"\nLast checkpoint:")
        print(f"  Epoch: {ckpt['epoch'] + 1}/{cfg.EPOCHS}")
        print(f"  Progress: {100 * (ckpt['epoch'] + 1) / cfg.EPOCHS:.1f}%")
        if 'metrics' in ckpt:
            print(f"  Train Loss: {ckpt['metrics'].get('train_loss', 'N/A')}")
            print(f"  Val Dice: {ckpt['metrics'].get('val_dice', 'N/A')}")
    else:
        print("\nNo checkpoint found. Training not started.")
    
    if best_ckpt.exists():
        ckpt = torch.load(best_ckpt, map_location='cpu', weights_only=False)
        print(f"\nBest model:")
        print(f"  Epoch: {ckpt['best_epoch'] + 1}")
        print(f"  Best Dice: {ckpt['best_score']:.4f}")
    
    if history_file.exists():
        with open(history_file, 'r') as f:
            history = json.load(f)
        print(f"\nTraining history: {len(history)} epochs recorded")
        if len(history) > 0:
            recent = history[-5:]
            print("\nRecent epochs:")
            for h in recent:
                print(f"  Epoch {h['epoch']+1}: loss={h.get('train_loss', 0):.4f}, "
                      f"val_dice={h.get('val_dice', 0):.4f}")
    
    print("="*60)

check_training_status()